# fm-value-audit — real-data Stage 1

**Purpose:** validate and standardize the uploaded public datasets before any foundation-model comparison or final holdout evaluation.

This notebook deliberately does **not** train, tune, rank targets, or inspect the final temporal holdout. It produces auditable inputs for the next phase:

- `GSE115978`: raw-count AnnData and sample pseudobulk counts.
- `GSE120575`: canonical metadata and sample-level mean TPM. TPM is **not** relabeled as raw counts and is not used as direct Geneformer/scGPT input.
- `GSE179994`: raw-count AnnData, official clinical-response mapping from the paper supplement when available, and a deterministic model-compute subset.
- Normal skin and lung controls from the versioned CZ CELLxGENE Census `stable` release.
- Checksums, software versions, alignment diagnostics, and a machine-readable audit report.

**Recommended runtime:** Colab high-RAM CPU for this stage. A GPU is unnecessary. Keep the original compressed files unchanged in Drive.

In [ ]:
# @title 1. Install Stage-1 dependencies
# A runtime restart is normally not required. If Colab explicitly asks for one,
# restart and run the notebook from the beginning.
!pip -q install --upgrade \
    "anndata>=0.10,<0.13" \
    "cellxgene-census>=1.17" \
    "openpyxl>=3.1" \
    "pyarrow>=15" \
    "requests>=2.31"

# R is used only to read the official sparse .rds object without converting it
# through an unverified third-party service.
!apt-get -qq update
!apt-get -qq install -y r-base-core r-cran-matrix

In [ ]:
# @title 2. Configuration
import gzip
import hashlib
import importlib.metadata
import json
import platform
import random
import re
import subprocess
import sys
import warnings
from dataclasses import asdict, dataclass
from datetime import UTC, datetime
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import requests
from scipy import sparse
from scipy.io import mmread

SEED = 20260729
random.seed(SEED)
np.random.seed(SEED)

PROJECT_FOLDER_NAME = "fm-value-audit-real"
RUN_GSE115978 = True
RUN_GSE120575 = True
RUN_GSE179994 = True
RUN_CELLXGENE_CONTROLS = True

# These caps limit later model cost; full standardized objects are still retained.
GSE179994_MODEL_SUBSET_CELLS = 20_000
CONTROL_CELLS_PER_TISSUE = 10_000
CONTROL_MAX_CELLS_PER_DONOR = 500

# Official supplementary workbook linked by the Nature Cancer article.
GSE179994_SUPPLEMENT_URL = (
    "https://media.springernature.com/original/springer-static/esm/"
    "art%3A10.1038%2Fs43018-021-00292-8/"
    "MediaObjects/43018_2021_292_MOESM3_ESM.xlsx"
)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Seed:", SEED)

In [ ]:
# @title 3. Mount Drive and locate the project
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")


def find_project_folder(root: Path, name: str) -> Path:
    direct = root / name
    if direct.is_dir():
        return direct
    matches = [p for p in root.rglob(name) if p.is_dir()]
    if not matches:
        raise FileNotFoundError(
            f"Could not find a folder named {name!r} under {root}. "
            "Add the shared folder to My Drive or move it under My Drive."
        )
    if len(matches) > 1:
        print("Multiple project folders found; using:", matches[0])
        for candidate in matches:
            print(" -", candidate)
    return matches[0]


PROJECT = find_project_folder(DRIVE_ROOT, PROJECT_FOLDER_NAME)
DATA = PROJECT / "data"
MODELS = PROJECT / "models"
OUTPUTS = PROJECT / "outputs" / "stage1"
OUTPUTS.mkdir(parents=True, exist_ok=True)

PATHS = {
    "gse115978_counts": DATA / "GSE115978" / "GSE115978_counts.csv.gz",
    "gse115978_annotations": DATA / "GSE115978" / "GSE115978_cell.annotations.csv.gz",
    "gse120575_tpm": DATA
    / "GSE120575"
    / "GSE120575_Sade_Feldman_melanoma_single_cells_TPM_GEO.txt.gz",
    "gse120575_metadata": DATA / "GSE120575" / "GSE120575_patient_ID_single_cells.txt.gz",
    "gse179994_counts_rds": DATA / "GSE179994" / "GSE179994_all.Tcell.rawCounts.rds.gz",
    "gse179994_metadata": DATA / "GSE179994" / "GSE179994_Tcell.metadata.tsv.gz",
    "gse179994_supplement": DATA / "GSE179994" / "GSE179994_supplementary_tables.xlsx",
    "geneformer_config": MODELS / "Geneformer-V1-10M" / "config.json",
    "geneformer_weights": MODELS / "Geneformer-V1-10M" / "model.safetensors",
    "geneformer_token_dict": MODELS / "gene_dictionaries_30m" / "token_dictionary_gc30M.pkl",
    "geneformer_gene_medians": MODELS
    / "gene_dictionaries_30m"
    / "gene_median_dictionary_gc30M.pkl",
    "scgpt_args": MODELS / "scGPT-whole-human" / "args.json",
    "scgpt_vocab": MODELS / "scGPT-whole-human" / "vocab.json",
    "scgpt_weights": MODELS / "scGPT-whole-human" / "best_model.pt",
}

print("Project:", PROJECT)
print("Output directory:", OUTPUTS)

In [ ]:
# @title 4. Audit helpers and immutable input inventory
@dataclass
class Check:
    name: str
    status: str
    detail: str


CHECKS: list[Check] = []
GENERATED: list[str] = []


def record(name: str, condition: bool, detail: str) -> None:
    status = "PASS" if condition else "FAIL"
    CHECKS.append(Check(name=name, status=status, detail=detail))
    print(f"[{status}] {name}: {detail}")
    if not condition:
        raise AssertionError(f"{name}: {detail}")


def warn_record(name: str, detail: str) -> None:
    CHECKS.append(Check(name=name, status="WARNING", detail=detail))
    warnings.warn(f"{name}: {detail}", stacklevel=2)


def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def file_record(path: Path, with_hash: bool = True) -> dict[str, object]:
    result: dict[str, object] = {
        "path": str(path.relative_to(PROJECT)),
        "exists": path.is_file(),
    }
    if path.is_file():
        result["bytes"] = path.stat().st_size
        result["sha256"] = sha256_file(path) if with_hash else "NOT_COMPUTED"
    return result


required = [
    "gse115978_counts",
    "gse115978_annotations",
    "gse120575_tpm",
    "gse120575_metadata",
    "gse179994_counts_rds",
    "gse179994_metadata",
    "geneformer_config",
    "geneformer_weights",
    "geneformer_token_dict",
    "geneformer_gene_medians",
    "scgpt_args",
    "scgpt_vocab",
    "scgpt_weights",
]
for key in required:
    record(f"input_exists::{key}", PATHS[key].is_file(), str(PATHS[key]))

# Hash all inputs, including the 441 MB RDS. This is I/O-heavy but creates a
# trustworthy immutable manifest and does not load the objects into memory.
input_manifest = {key: file_record(PATHS[key]) for key in required}
manifest_path = OUTPUTS / "input_manifest.json"
manifest_path.write_text(json.dumps(input_manifest, indent=2), encoding="utf-8")
GENERATED.append(str(manifest_path.relative_to(PROJECT)))
print("Wrote", manifest_path)

## Dataset policy fixed before processing

- **GSE115978:** raw counts; suitable for model tokenization after identifier mapping. It has treatment-status annotations, not a complete clinical-response label. No response is inferred.
- **GSE120575:** TPM; suitable for sample-level expression summaries and external validation, but not passed off as raw counts.
- **GSE179994:** raw T-cell counts; suitable for model tokenization. Clinical response is taken only from the official publication supplement and retained with provenance.
- **Normal controls:** versioned, primary human cells from CELLxGENE Census. Sampling is deterministic and donor-capped.

In [ ]:
# @title 5. GSE115978 — raw counts to full and pseudobulk AnnData


def build_gse115978() -> dict[str, object]:
    counts_path = PATHS["gse115978_counts"]
    annotation_path = PATHS["gse115978_annotations"]

    obs = pd.read_csv(annotation_path)
    record(
        "GSE115978_annotation_unique_cells", obs["cells"].is_unique, f"{len(obs):,} annotation rows"
    )

    header = pd.read_csv(counts_path, nrows=0).columns.tolist()
    count_cells = header[1:]
    record(
        "GSE115978_cell_count_alignment",
        len(count_cells) == len(obs),
        f"matrix={len(count_cells):,}; metadata={len(obs):,}",
    )
    record(
        "GSE115978_cell_order_alignment",
        count_cells == obs["cells"].tolist(),
        "matrix columns exactly match metadata order",
    )

    gene_ids: list[str] = []
    blocks: list[sparse.csr_matrix] = []
    for chunk_i, chunk in enumerate(pd.read_csv(counts_path, index_col=0, chunksize=500)):
        genes = chunk.index.astype(str).tolist()
        if len(set(genes)) != len(genes):
            raise ValueError(f"Duplicate genes inside chunk {chunk_i}")
        gene_ids.extend(genes)
        values = chunk.to_numpy(dtype=np.int32, copy=False)
        if np.any(values < 0):
            raise ValueError("Raw-count matrix contains negative values")
        blocks.append(sparse.csr_matrix(values))
        if chunk_i % 10 == 0:
            print(f"  read {len(gene_ids):,} genes")

    gene_by_cell = sparse.vstack(blocks, format="csr")
    del blocks
    record(
        "GSE115978_unique_genes", len(gene_ids) == len(set(gene_ids)), f"{len(gene_ids):,} genes"
    )
    record(
        "GSE115978_matrix_shape",
        gene_by_cell.shape == (len(gene_ids), len(obs)),
        str(gene_by_cell.shape),
    )

    X = gene_by_cell.T.tocsr()
    obs = obs.set_index("cells", drop=False)
    var = pd.DataFrame(index=pd.Index(gene_ids, name="gene_symbol"))
    var["gene_symbol"] = var.index

    adata = ad.AnnData(X=X, obs=obs, var=var)
    adata.uns["provenance"] = {
        "accession": "GSE115978",
        "matrix_semantics": "raw_counts",
        "source_files": [counts_path.name, annotation_path.name],
        "seed": SEED,
    }
    full_path = OUTPUTS / "GSE115978_raw_counts.h5ad"
    adata.write_h5ad(full_path, compression="gzip")
    GENERATED.append(str(full_path.relative_to(PROJECT)))

    # True count aggregation by biological sample; no cell-level pseudoreplication.
    sample_categories = pd.Categorical(obs["samples"])
    rows = np.arange(adata.n_obs)
    membership = sparse.csr_matrix(
        (np.ones(adata.n_obs, dtype=np.int8), (rows, sample_categories.codes)),
        shape=(adata.n_obs, len(sample_categories.categories)),
    )
    pb_X = (membership.T @ adata.X).tocsr()
    sample_obs = (
        obs.reset_index(drop=True)
        .groupby("samples", sort=False, observed=True)
        .agg(
            treatment_group=("treatment.group", "first"),
            cohort=("Cohort", "first"),
            n_cells=("cells", "size"),
        )
        .reindex(sample_categories.categories)
    )
    sample_obs.index.name = "sample_id"
    pb = ad.AnnData(X=pb_X, obs=sample_obs, var=var.copy())
    pb.uns["provenance"] = {
        "accession": "GSE115978",
        "aggregation": "sum_raw_counts_by_sample",
        "clinical_response_available": False,
    }
    pb_path = OUTPUTS / "GSE115978_sample_pseudobulk_counts.h5ad"
    pb.write_h5ad(pb_path, compression="gzip")
    GENERATED.append(str(pb_path.relative_to(PROJECT)))

    summary = {
        "cells": int(adata.n_obs),
        "genes": int(adata.n_vars),
        "samples": int(pb.n_obs),
        "nonzero_entries": int(adata.X.nnz),
        "cell_types": obs["cell.types"].value_counts().to_dict(),
        "treatment_groups": obs["treatment.group"].value_counts().to_dict(),
        "clinical_response_available": False,
    }
    summary_path = OUTPUTS / "GSE115978_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    GENERATED.append(str(summary_path.relative_to(PROJECT)))
    return summary


GSE115978_SUMMARY = build_gse115978() if RUN_GSE115978 else {"status": "SKIPPED"}
GSE115978_SUMMARY

In [ ]:
# @title 6. GSE120575 — canonical metadata and patient-state mean TPM

import csv


def parse_gse120575_metadata(path: Path) -> pd.DataFrame:
    rows: list[list[str]] = []
    header: list[str] | None = None
    with gzip.open(path, "rt", encoding="latin1", newline="") as handle:
        for row in csv.reader(handle, delimiter="\t"):
            if row and row[0] == "Sample name":
                header = row
                continue
            if header is not None and row and re.fullmatch(r"Sample \d+", row[0]):
                padded = row + [""] * (len(header) - len(row))
                rows.append(padded[: len(header)])
    if header is None or not rows:
        raise ValueError("Could not locate biological sample rows")
    table = pd.DataFrame(rows, columns=header)
    table = table.loc[:, ~table.columns.duplicated()]
    table = table.rename(
        columns={
            "Sample name": "geo_sample_name",
            "title": "cell_id",
            "characteristics: patinet ID (Pre=baseline; Post= on treatment)": "patient_state",
            "characteristics: response": "response",
            "characteristics: therapy": "therapy",
        }
    )
    required_cols = ["geo_sample_name", "cell_id", "patient_state", "response", "therapy"]
    missing = [c for c in required_cols if c not in table.columns]
    if missing:
        raise ValueError(f"Missing expected GSE120575 metadata columns: {missing}")
    table = table[required_cols].copy()
    extracted = table["patient_state"].str.extract(r"^(Pre|Post)_(.+)$")
    table["timepoint"] = extracted[0]
    table["patient_id"] = extracted[1]
    table["accession"] = "GSE120575"
    return table


def build_gse120575() -> dict[str, object]:
    tpm_path = PATHS["gse120575_tpm"]
    meta = parse_gse120575_metadata(PATHS["gse120575_metadata"])
    record("GSE120575_unique_cell_ids", meta["cell_id"].is_unique, f"{len(meta):,} cells")
    record(
        "GSE120575_response_nonmissing",
        meta["response"].notna().all(),
        str(meta["response"].value_counts().to_dict()),
    )

    with gzip.open(tpm_path, "rt", encoding="utf-8") as handle:
        matrix_cells = handle.readline().rstrip("\n\r").split("\t")[1:]
    record(
        "GSE120575_cell_count_alignment",
        len(matrix_cells) == len(meta),
        f"matrix={len(matrix_cells):,}; metadata={len(meta):,}",
    )

    meta_by_cell = meta.set_index("cell_id")
    missing_in_meta = sorted(set(matrix_cells) - set(meta_by_cell.index))
    missing_in_matrix = sorted(set(meta_by_cell.index) - set(matrix_cells))
    record(
        "GSE120575_matrix_cells_in_metadata", not missing_in_meta, f"missing={len(missing_in_meta)}"
    )
    record(
        "GSE120575_metadata_cells_in_matrix",
        not missing_in_matrix,
        f"missing={len(missing_in_matrix)}",
    )
    aligned_meta = meta_by_cell.loc[matrix_cells].reset_index()

    states = sorted(aligned_meta["patient_state"].astype(str).unique())
    state_lookup = {value: index for index, value in enumerate(states)}
    state_index = np.asarray(
        [state_lookup[value] for value in aligned_meta["patient_state"].astype(str)],
        dtype=np.int64,
    )
    state_sizes = np.bincount(state_index, minlength=len(states)).astype(np.float64)

    gene_ids: list[str] = []
    state_sum_rows: list[np.ndarray] = []
    global_mean: list[float] = []
    fraction_positive: list[float] = []

    # The first line is the cell header, the second is an embedded patient-state
    # vector, and every gene row contains one verified trailing empty field.
    with gzip.open(tpm_path, "rb") as handle:
        handle.readline()
        handle.readline()
        for line_i, raw_line in enumerate(handle, start=1):
            if not raw_line.strip():
                continue
            gene_raw, values_raw = raw_line.split(b"\t", 1)
            values = np.fromstring(values_raw, sep="\t", dtype=np.float32)
            if len(values) != len(matrix_cells):
                raise ValueError(
                    f"Row {gene_raw!r} has {len(values)} values; expected {len(matrix_cells)}"
                )
            gene_ids.append(gene_raw.decode("utf-8"))
            state_sum_rows.append(np.bincount(state_index, weights=values, minlength=len(states)))
            global_mean.append(float(values.mean(dtype=np.float64)))
            fraction_positive.append(float(np.count_nonzero(values) / len(values)))
            if line_i % 5000 == 0:
                print(f"  scanned {line_i:,} rows; retained {len(gene_ids):,} genes")

    record(
        "GSE120575_unique_genes", len(gene_ids) == len(set(gene_ids)), f"{len(gene_ids):,} genes"
    )
    state_means = np.vstack(state_sum_rows) / np.maximum(state_sizes, 1.0)[None, :]
    sample_mean_df = pd.DataFrame(state_means, index=gene_ids, columns=states)
    sample_mean_df.index.name = "gene_symbol"
    sample_mean_path = OUTPUTS / "GSE120575_sample_mean_TPM.parquet"
    sample_mean_df.to_parquet(sample_mean_path)
    GENERATED.append(str(sample_mean_path.relative_to(PROJECT)))

    gene_stats = pd.DataFrame(
        {
            "gene_symbol": gene_ids,
            "mean_TPM_all_cells": global_mean,
            "fraction_cells_TPM_gt_0": fraction_positive,
        }
    )
    stats_path = OUTPUTS / "GSE120575_gene_stats.parquet"
    gene_stats.to_parquet(stats_path, index=False)
    GENERATED.append(str(stats_path.relative_to(PROJECT)))

    sample_meta = (
        aligned_meta.groupby("patient_state", sort=True)
        .agg(
            patient_id=("patient_id", "first"),
            timepoint=("timepoint", "first"),
            response=("response", "first"),
            therapy=("therapy", "first"),
            n_cells=("cell_id", "size"),
        )
        .reindex(states)
        .reset_index()
    )
    sample_meta_path = OUTPUTS / "GSE120575_sample_metadata.csv"
    sample_meta.to_csv(sample_meta_path, index=False)
    GENERATED.append(str(sample_meta_path.relative_to(PROJECT)))

    cell_meta_path = OUTPUTS / "GSE120575_cell_metadata.parquet"
    aligned_meta.to_parquet(cell_meta_path, index=False)
    GENERATED.append(str(cell_meta_path.relative_to(PROJECT)))

    summary = {
        "cells": len(aligned_meta),
        "genes": len(gene_ids),
        "patient_states": int(sample_meta.shape[0]),
        "patients": int(aligned_meta["patient_id"].nunique()),
        "responses_by_cell": aligned_meta["response"].value_counts().to_dict(),
        "therapies_by_cell": aligned_meta["therapy"].value_counts().to_dict(),
        "matrix_semantics": "TPM; not raw counts; excluded from direct model tokenization",
    }
    summary_path = OUTPUTS / "GSE120575_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    GENERATED.append(str(summary_path.relative_to(PROJECT)))
    return summary


GSE120575_SUMMARY = build_gse120575() if RUN_GSE120575 else {"status": "SKIPPED"}
GSE120575_SUMMARY

In [ ]:
# @title 7. GSE179994 — retrieve and inspect the official supplementary workbook


def download_with_checks(url: str, destination: Path) -> None:
    if destination.is_file() and destination.stat().st_size > 10_000:
        print("Using existing supplement:", destination)
        return
    response = requests.get(
        url,
        timeout=180,
        headers={"User-Agent": "Mozilla/5.0 (compatible; fm-value-audit/1.0)"},
    )
    response.raise_for_status()
    destination.write_bytes(response.content)
    if destination.stat().st_size < 10_000:
        raise RuntimeError("Downloaded supplementary workbook is unexpectedly small")
    print("Downloaded", destination, destination.stat().st_size, "bytes")


def normalized_column(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")


def normalize_response(value: object) -> str | None:
    if pd.isna(value):
        return None
    raw = str(value).strip()
    text = re.sub(r"[^A-Z0-9]+", " ", raw.upper()).strip()
    # Conservative mapping only for unambiguous labels.
    if text in {"NR", "N", "NO"}:
        return "Non-responder"
    if text in {"R", "Y", "YES"}:
        return "Responder"
    if re.search(r"NON ?RESP|NO RESPONSE|PROGRESSIVE DISEASE|\bPD\b|PROGRESSOR", text):
        return "Non-responder"
    if re.search(r"COMPLETE RESPONSE|PARTIAL RESPONSE|\bCR\b|\bPR\b|\bRESPONDER\b", text):
        return "Responder"
    return None


def normalize_sample_id(value: object) -> str:
    text = str(value).strip().lower()
    text = text.replace("pretreatment", "pre").replace("pre-treatment", "pre")
    text = text.replace("posttreatment", "post").replace("post-treatment", "post")
    text = re.sub(r"[_\-/\s]+", ".", text)
    text = re.sub(r"\.+", ".", text).strip(".")
    return text


def discover_response_tables(workbook: Path) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    xls = pd.ExcelFile(workbook)
    candidates: list[dict[str, object]] = []
    candidate_frames: list[tuple[tuple[int, int, int], pd.DataFrame]] = []
    for sheet in xls.sheet_names:
        raw = pd.read_excel(workbook, sheet_name=sheet, header=None)
        max_header = min(30, len(raw))
        for header_row in range(max_header):
            headers = [normalized_column(x) for x in raw.iloc[header_row].tolist()]
            has_patient_or_sample = any(
                re.search(r"patient|sample|biopsy|subject", h) for h in headers
            )
            has_response = any(
                re.search(r"response|responder|outcome|best_response", h) for h in headers
            )
            if not (has_patient_or_sample and has_response):
                continue
            df = pd.read_excel(workbook, sheet_name=sheet, header=header_row)
            df.columns = [normalized_column(c) for c in df.columns]
            sample_cols = [c for c in df.columns if re.search(r"sample|biopsy", c)]
            patient_cols = [c for c in df.columns if re.search(r"patient|subject", c)]
            time_cols = [c for c in df.columns if re.search(r"time|visit|pre|post", c)]
            response_cols = [
                c for c in df.columns if re.search(r"response|responder|outcome|best_response", c)
            ]
            if not response_cols:
                continue
            response_col = response_cols[0]
            score = (
                4 * len(response_cols) + 2 * len(sample_cols) + len(patient_cols) + len(time_cols)
            )
            keep = [
                c
                for c in [*(sample_cols[:1]), *(patient_cols[:1]), *(time_cols[:1]), response_col]
                if c
            ]
            small = df[keep].copy()
            small["source_sheet"] = sheet
            small["source_header_row_zero_based"] = header_row
            small["raw_response"] = df[response_col]
            small["response_normalized"] = df[response_col].map(normalize_response)
            n_normalized = int(small["response_normalized"].notna().sum())
            n_rows = len(small)
            candidate = {
                "sheet": sheet,
                "header_row_zero_based": header_row,
                "columns": list(df.columns),
                "score": score,
                "normalized_response_rows": n_normalized,
                "rows": n_rows,
            }
            candidates.append(candidate)
            candidate_frames.append(((n_normalized, score, n_rows), small))
    if not candidate_frames:
        return pd.DataFrame(), candidates
    # Use one best-supported table only. Combining several plausible header rows
    # would duplicate patients and can create artificial response conflicts.
    _, best = max(candidate_frames, key=lambda item: item[0])
    return best.reset_index(drop=True), candidates


supp_path = PATHS["gse179994_supplement"]
download_with_checks(GSE179994_SUPPLEMENT_URL, supp_path)
GENERATED.append(str(supp_path.relative_to(PROJECT)))

response_candidates, supplement_inventory = discover_response_tables(supp_path)
inv_path = OUTPUTS / "GSE179994_supplement_inventory.json"
inv_path.write_text(json.dumps(supplement_inventory, indent=2, default=str), encoding="utf-8")
GENERATED.append(str(inv_path.relative_to(PROJECT)))

candidate_path = OUTPUTS / "GSE179994_response_candidates.csv"
response_candidates.to_csv(candidate_path, index=False)
GENERATED.append(str(candidate_path.relative_to(PROJECT)))
print("Candidate rows:", len(response_candidates))
print("Candidate tables:", len(supplement_inventory))
response_candidates.head(20)

In [ ]:
# @title 8. GSE179994 — convert the official R sparse matrix to Matrix Market


def convert_rds_to_matrix_market(rds_path: Path, output_prefix: Path) -> dict[str, Path]:
    mtx = output_prefix.with_suffix(".mtx.gz")
    rows = output_prefix.with_name(output_prefix.name + "_rownames.txt.gz")
    cols = output_prefix.with_name(output_prefix.name + "_colnames.txt.gz")
    info = output_prefix.with_name(output_prefix.name + "_r_object_info.txt")
    script = output_prefix.with_name(output_prefix.name + "_convert.R")

    if all(path.is_file() and path.stat().st_size > 0 for path in [mtx, rows, cols, info]):
        print("Using existing Matrix Market conversion")
        return {"mtx": mtx, "rows": rows, "cols": cols, "info": info}

    r_code = r"""
args <- commandArgs(trailingOnly = TRUE)
infile <- args[[1]]
out_mtx <- args[[2]]
out_rows <- args[[3]]
out_cols <- args[[4]]
out_info <- args[[5]]
suppressPackageStartupMessages(library(Matrix))
obj <- readRDS(gzfile(infile, open = "rb"))
info <- c(
  paste0("class=", paste(class(obj), collapse=",")),
  paste0("dim=", paste(dim(obj), collapse="x"))
)
if (is.data.frame(obj)) obj <- as.matrix(obj)
if (is.matrix(obj)) obj <- Matrix(obj, sparse = TRUE)
if (!inherits(obj, "sparseMatrix")) {
  stop(paste("Expected matrix-like R object; observed class", paste(class(obj), collapse=",")))
}
if (is.null(rownames(obj)) || is.null(colnames(obj))) {
  stop("The R matrix must contain row and column names")
}
con <- gzfile(out_mtx, open = "wt")
writeMM(obj, file = con)
close(con)
row_con <- gzfile(out_rows, open = "wt")
writeLines(rownames(obj), con = row_con, useBytes = TRUE)
close(row_con)
col_con <- gzfile(out_cols, open = "wt")
writeLines(colnames(obj), con = col_con, useBytes = TRUE)
close(col_con)
writeLines(info, con = out_info)
"""
    script.write_text(r_code, encoding="utf-8")
    cmd = ["Rscript", str(script), str(rds_path), str(mtx), str(rows), str(cols), str(info)]
    print("Running:", " ".join(cmd[:2]), "...")
    subprocess.run(cmd, check=True)
    for path in [mtx, rows, cols, info]:
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f"R conversion did not produce {path}")
    return {"mtx": mtx, "rows": rows, "cols": cols, "info": info}


RDS_EXPORT = (
    convert_rds_to_matrix_market(PATHS["gse179994_counts_rds"], OUTPUTS / "GSE179994_rawCounts")
    if RUN_GSE179994
    else {}
)
RDS_EXPORT

In [ ]:
# @title 9. GSE179994 — align cells, attach response, and write AnnData


def read_gzip_lines(path: Path) -> list[str]:
    with gzip.open(path, "rt", encoding="utf-8", errors="strict") as handle:
        return [line.rstrip("\n\r") for line in handle]


def build_patient_response_map(candidates: pd.DataFrame) -> pd.DataFrame:
    if candidates.empty or "response_normalized" not in candidates:
        return pd.DataFrame(
            columns=["patient", "response_normalized", "raw_response", "source_sheet"]
        )
    valid = candidates[candidates["response_normalized"].notna()].copy()
    patient_cols = [c for c in valid.columns if re.search(r"patient|subject", c)]
    sample_cols = [c for c in valid.columns if re.search(r"sample|biopsy", c)]
    source_col = patient_cols[0] if patient_cols else (sample_cols[0] if sample_cols else None)
    if source_col is None:
        return pd.DataFrame(
            columns=["patient", "response_normalized", "raw_response", "source_sheet"]
        )
    patient_number = valid[source_col].astype(str).str.extract(r"[Pp]\s*0*(\d+)", expand=False)
    valid["patient"] = patient_number.map(lambda x: f"P{int(x)}" if pd.notna(x) else None)
    valid = valid.dropna(subset=["patient"])
    # Preserve disagreements rather than silently choosing one.
    conflicts = valid.groupby("patient")["response_normalized"].nunique()
    conflict_patients = conflicts[conflicts > 1].index.tolist()
    if conflict_patients:
        warn_record(
            "GSE179994_response_conflicts",
            f"Conflicting normalized responses for: {conflict_patients}",
        )
        valid = valid[~valid["patient"].isin(conflict_patients)]
    return valid.sort_values(["patient", "source_sheet"]).drop_duplicates("patient")[
        ["patient", "response_normalized", "raw_response", "source_sheet"]
    ]


def deterministic_balanced_subset(
    obs: pd.DataFrame, max_cells: int, strata: list[str]
) -> np.ndarray:
    if len(obs) <= max_cells:
        return np.arange(len(obs), dtype=int)
    work = obs.reset_index(drop=True).copy()
    work["_row"] = np.arange(len(work))
    for col in strata:
        work[col] = work[col].fillna("<NA>").astype(str)
    work["_stratum"] = work[strata].agg("|".join, axis=1)
    groups = list(work.groupby("_stratum", sort=True))
    per_group = max(1, max_cells // len(groups))
    chosen: list[int] = []
    rng = np.random.default_rng(SEED)
    leftovers: list[int] = []
    for _, group in groups:
        idx = group["_row"].to_numpy()
        idx = rng.permutation(idx)
        chosen.extend(idx[:per_group].tolist())
        leftovers.extend(idx[per_group:].tolist())
    if len(chosen) < max_cells and leftovers:
        chosen.extend(rng.permutation(np.asarray(leftovers))[: max_cells - len(chosen)].tolist())
    return np.asarray(sorted(chosen[:max_cells]), dtype=int)


def build_gse179994() -> dict[str, object]:
    meta = pd.read_csv(PATHS["gse179994_metadata"], sep="\t", dtype=str)
    record(
        "GSE179994_unique_metadata_cells", meta["cellid"].is_unique, f"{len(meta):,} metadata cells"
    )

    row_names = read_gzip_lines(RDS_EXPORT["rows"])
    col_names = read_gzip_lines(RDS_EXPORT["cols"])
    with gzip.open(RDS_EXPORT["mtx"], "rb") as handle:
        matrix = mmread(handle).tocsr()
    record(
        "GSE179994_export_shape",
        matrix.shape == (len(row_names), len(col_names)),
        f"matrix={matrix.shape}; names=({len(row_names)}, {len(col_names)})",
    )

    meta_cells = set(meta["cellid"])
    row_overlap = len(meta_cells.intersection(row_names))
    col_overlap = len(meta_cells.intersection(col_names))
    print("Metadata overlap — rows:", row_overlap, "columns:", col_overlap)

    if col_overlap >= row_overlap and col_overlap > 0:
        genes = row_names
        cells = col_names
        X = matrix.T.tocsr()
    elif row_overlap > 0:
        cells = row_names
        genes = col_names
        X = matrix.tocsr()
    else:
        raise ValueError("Could not align GSE179994 metadata IDs to either matrix axis")

    record(
        "GSE179994_unique_matrix_cells",
        len(cells) == len(set(cells)),
        f"{len(cells):,} matrix cells",
    )
    record("GSE179994_unique_genes", len(genes) == len(set(genes)), f"{len(genes):,} genes")

    matrix_index = pd.Index(cells)
    meta_index = pd.Index(meta["cellid"])
    missing_meta = meta_index.difference(matrix_index)
    missing_matrix = matrix_index.difference(meta_index)
    record(
        "GSE179994_all_metadata_cells_in_matrix",
        len(missing_meta) == 0,
        f"missing metadata cells={len(missing_meta)}",
    )
    if len(missing_matrix):
        warn_record(
            "GSE179994_matrix_cells_without_metadata",
            f"{len(missing_matrix)} cells will be excluded",
        )

    positions = matrix_index.get_indexer(meta_index)
    X = X[positions, :].tocsr()
    obs = meta.set_index("cellid", drop=False)

    patient_response = build_patient_response_map(response_candidates)
    response_map_path = OUTPUTS / "GSE179994_patient_response_mapping.csv"
    patient_response.to_csv(response_map_path, index=False)
    GENERATED.append(str(response_map_path.relative_to(PROJECT)))
    response_lookup = patient_response.set_index("patient")["response_normalized"].to_dict()
    obs["clinical_response"] = obs["patient"].map(response_lookup)
    obs["timepoint"] = np.where(obs["sample"].str.contains(".pre", regex=False), "pre", "post")

    var = pd.DataFrame(index=pd.Index(genes, name="gene_id"))
    var["gene_id"] = var.index
    adata = ad.AnnData(X=X, obs=obs, var=var)
    adata.uns["provenance"] = {
        "accession": "GSE179994",
        "matrix_semantics": "raw_counts",
        "response_source": "official Nature Cancer supplementary workbook",
        "response_mapping_coverage_patients": int(patient_response["patient"].nunique()),
        "seed": SEED,
    }
    full_path = OUTPUTS / "GSE179994_Tcell_raw_counts.h5ad"
    adata.write_h5ad(full_path, compression="gzip")
    GENERATED.append(str(full_path.relative_to(PROJECT)))

    subset_idx = deterministic_balanced_subset(
        obs.reset_index(drop=True),
        max_cells=GSE179994_MODEL_SUBSET_CELLS,
        strata=["sample", "celltype", "cluster"],
    )
    subset = adata[subset_idx].copy()
    subset.uns["subset_policy"] = {
        "method": "deterministic_balanced_by_sample_celltype_cluster",
        "max_cells": GSE179994_MODEL_SUBSET_CELLS,
        "seed": SEED,
    }
    subset_path = OUTPUTS / "GSE179994_Tcell_model_subset.h5ad"
    subset.write_h5ad(subset_path, compression="gzip")
    GENERATED.append(str(subset_path.relative_to(PROJECT)))

    summary = {
        "cells": int(adata.n_obs),
        "genes": int(adata.n_vars),
        "patients": int(obs["patient"].nunique()),
        "samples": int(obs["sample"].nunique()),
        "model_subset_cells": int(subset.n_obs),
        "patients_with_response_mapping": int(patient_response["patient"].nunique()),
        "cells_with_response_mapping": int(obs["clinical_response"].notna().sum()),
        "response_counts_by_cell": obs["clinical_response"]
        .value_counts(dropna=False)
        .astype(int)
        .to_dict(),
        "matrix_nonzero_entries": int(adata.X.nnz),
    }
    summary_path = OUTPUTS / "GSE179994_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
    GENERATED.append(str(summary_path.relative_to(PROJECT)))
    return summary


GSE179994_SUMMARY = build_gse179994() if RUN_GSE179994 else {"status": "SKIPPED"}
GSE179994_SUMMARY

In [ ]:
# @title 10. Versioned normal skin and lung controls from CELLxGENE Census


def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "NOT_INSTALLED"


def donor_capped_selection(obs: pd.DataFrame, max_total: int, max_per_donor: int) -> np.ndarray:
    work = obs.copy()
    work["donor_key"] = (
        work["dataset_id"].fillna("<NA>").astype(str)
        + "|"
        + work["donor_id"].fillna("<NA>").astype(str)
    )
    work = work.sort_values(["donor_key", "soma_joinid"], kind="mergesort")
    selected = work.groupby("donor_key", sort=True).head(max_per_donor)
    if len(selected) > max_total:
        selected = selected.sample(n=max_total, random_state=SEED).sort_values("soma_joinid")
    return selected["soma_joinid"].to_numpy(dtype=np.int64)


def fetch_control(tissue_general: str, label: str) -> dict[str, object]:
    import cellxgene_census

    obs_filter = (
        f"is_primary_data == True and disease == 'normal' and tissue_general == '{tissue_general}'"
    )
    with cellxgene_census.open_soma(census_version="stable") as census:
        metadata = cellxgene_census.get_obs(
            census,
            organism="Homo sapiens",
            value_filter=obs_filter,
            column_names=[
                "soma_joinid",
                "dataset_id",
                "donor_id",
                "assay",
                "cell_type",
                "disease",
                "tissue",
                "tissue_general",
                "sex",
                "suspension_type",
                "is_primary_data",
            ],
        )
        if metadata.empty:
            raise RuntimeError(f"No normal primary cells found for {tissue_general}")
        selected_joinids = donor_capped_selection(
            metadata, CONTROL_CELLS_PER_TISSUE, CONTROL_MAX_CELLS_PER_DONOR
        )
        control = cellxgene_census.get_anndata(
            census,
            organism="Homo sapiens",
            X_name="raw",
            obs_coords=selected_joinids,
            obs_column_names=[
                "dataset_id",
                "donor_id",
                "assay",
                "cell_type",
                "disease",
                "tissue",
                "tissue_general",
                "sex",
                "suspension_type",
                "is_primary_data",
            ],
            var_column_names=["feature_id", "feature_name"],
        )

    control.uns["sampling"] = {
        "census_version_requested": "stable",
        "filter": obs_filter,
        "max_total_cells": CONTROL_CELLS_PER_TISSUE,
        "max_cells_per_dataset_donor": CONTROL_MAX_CELLS_PER_DONOR,
        "seed": SEED,
    }
    out = OUTPUTS / f"CELLxGENE_normal_{label}_raw_counts.h5ad"
    control.write_h5ad(out, compression="gzip")
    GENERATED.append(str(out.relative_to(PROJECT)))
    return {
        "label": label,
        "tissue_general": tissue_general,
        "available_cells_before_sampling": len(metadata),
        "selected_cells": int(control.n_obs),
        "genes": int(control.n_vars),
        "donors": int(control.obs[["dataset_id", "donor_id"]].drop_duplicates().shape[0]),
        "datasets": int(control.obs["dataset_id"].nunique()),
    }


CELLXGENE_SUMMARY: dict[str, object]
if RUN_CELLXGENE_CONTROLS:
    import cellxgene_census

    try:
        census_directory = cellxgene_census.get_census_version_directory()
        census_directory_serializable = json.loads(json.dumps(census_directory, default=str))
    except Exception as exc:
        census_directory_serializable = {"error": repr(exc), "requested": "stable"}

    CELLXGENE_SUMMARY = {
        "skin": fetch_control("skin of body", "skin"),
        "lung": fetch_control("lung", "lung"),
        "version_directory": census_directory_serializable,
        "client_version": package_version("cellxgene-census"),
    }
    census_path = OUTPUTS / "CELLxGENE_controls_summary.json"
    census_path.write_text(json.dumps(CELLXGENE_SUMMARY, indent=2, default=str), encoding="utf-8")
    GENERATED.append(str(census_path.relative_to(PROJECT)))
else:
    CELLXGENE_SUMMARY = {"status": "SKIPPED"}

CELLXGENE_SUMMARY

In [ ]:
# @title 11. Validate model files without running either model
with PATHS["geneformer_config"].open() as handle:
    geneformer_config = json.load(handle)
with PATHS["scgpt_args"].open() as handle:
    scgpt_args = json.load(handle)
with PATHS["scgpt_vocab"].open() as handle:
    scgpt_vocab = json.load(handle)

model_readiness = {
    "Geneformer-V1-10M": {
        "status": "FILES_VALIDATED_NOT_EXECUTED",
        "weights_bytes": PATHS["geneformer_weights"].stat().st_size,
        "hidden_size": geneformer_config.get("hidden_size"),
        "num_hidden_layers": geneformer_config.get("num_hidden_layers"),
        "num_attention_heads": geneformer_config.get("num_attention_heads"),
        "max_position_embeddings": geneformer_config.get("max_position_embeddings"),
        "vocab_size_config": geneformer_config.get("vocab_size"),
        "required_v1_policy": {
            "special_token": False,
            "model_input_size": 2048,
            "correct_30m_token_and_gene_median_dictionaries": True,
        },
    },
    "scGPT-whole-human": {
        "status": "FILES_VALIDATED_NOT_EXECUTED",
        "weights_bytes": PATHS["scgpt_weights"].stat().st_size,
        "nlayers": scgpt_args.get("nlayers"),
        "nheads": scgpt_args.get("nheads"),
        "embedding_size": scgpt_args.get("embsize"),
        "max_sequence_length": scgpt_args.get("max_seq_len"),
        "vocab_entries": len(scgpt_vocab),
    },
}
model_path = OUTPUTS / "model_readiness.json"
model_path.write_text(json.dumps(model_readiness, indent=2), encoding="utf-8")
GENERATED.append(str(model_path.relative_to(PROJECT)))
model_readiness

In [ ]:
# @title 12. Final Stage-1 audit report
software = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {
        name: package_version(name)
        for name in [
            "anndata",
            "cellxgene-census",
            "numpy",
            "pandas",
            "pyarrow",
            "requests",
            "scipy",
            "openpyxl",
        ]
    },
}

stage1_report = {
    "schema_version": "fmva-real-stage1-v1",
    "created_at_utc": datetime.now(UTC).isoformat(),
    "seed": SEED,
    "project_folder": str(PROJECT),
    "scientific_guardrails": {
        "final_temporal_holdout_touched": False,
        "foundation_models_executed": False,
        "model_tuning_performed": False,
        "GSE120575_semantics": "TPM, never represented as raw counts",
        "GSE115978_response_inferred": False,
        "GSE179994_response_source": "official publication supplement only",
    },
    "datasets": {
        "GSE115978": GSE115978_SUMMARY,
        "GSE120575": GSE120575_SUMMARY,
        "GSE179994": GSE179994_SUMMARY,
        "CELLxGENE_controls": CELLXGENE_SUMMARY,
    },
    "model_readiness": model_readiness,
    "checks": [asdict(check) for check in CHECKS],
    "generated_files": sorted(set(GENERATED)),
    "software": software,
    "next_phase": {
        "name": "Stage 2 — identifier harmonization and frozen embeddings",
        "allowed_after": "All Stage-1 checks pass and this report is reviewed",
        "not_yet_allowed": [
            "final target-ranking comparison",
            "temporal holdout evaluation",
            "README headline replacement",
        ],
    },
}

report_path = OUTPUTS / "stage1_audit_report.json"
report_path.write_text(json.dumps(stage1_report, indent=2, default=str), encoding="utf-8")
print("Wrote:", report_path)
print("Checks:", pd.Series([c.status for c in CHECKS]).value_counts().to_dict())
print("Generated files:", len(stage1_report["generated_files"]))

# Small human-readable completion marker that the Drive connector can inspect.
completion = OUTPUTS / "STAGE1_COMPLETE.txt"
completion.write_text(
    "fm-value-audit real-data Stage 1 completed\n"
    f"Created at UTC: {stage1_report['created_at_utc']}\n"
    f"Checks: {pd.Series([c.status for c in CHECKS]).value_counts().to_dict()}\n"
    "Foundation models executed: no\n"
    "Final temporal holdout touched: no\n"
    "See stage1_audit_report.json for full details.\n",
    encoding="utf-8",
)
print(completion.read_text())

## After this notebook finishes

The small files `outputs/stage1/STAGE1_COMPLETE.txt`, `stage1_audit_report.json`, and the dataset summaries are sufficient for an external audit. Do not manually edit them.

The next notebook will:

1. harmonize gene symbols and Ensembl IDs against the exact Geneformer/scGPT vocabularies;
2. freeze deterministic discovery, OOD, and random-control subsets;
3. run Geneformer V1-10M, scGPT whole-human, and architecture-matched random initialization with effort logging;
4. leave the final temporal target holdout untouched until all tuning decisions are frozen.